# 🔬 Smart Skin Care Advisor — Model Training Notebook

This notebook trains the SkinCare CNN on the dataset and evaluates its performance.

**Dataset expected structure:**
```
dataset/
  acne/
  eczema/
  melanoma/
  psoriasis/
  normal_skin/
  dark_spots/
  dry_skin/
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../ml'))

import torch
import matplotlib.pyplot as plt
import numpy as np

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────
DATASET_DIR = '../dataset'
BATCH_SIZE  = 32
EPOCHS      = 30
LR          = 1e-3
MODEL_OUT   = '../ml/skin_cnn.pth'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
from dataset_loader import get_dataloaders

train_loader, val_loader, class_names = get_dataloaders(
    DATASET_DIR, batch_size=BATCH_SIZE
)
print(f'Classes: {class_names}')
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# ── Visualise a batch ─────────────────────────────────────────────
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i >= len(images): break
    img = images[i].permute(1,2,0).numpy()
    img = (img * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)
    ax.imshow(img)
    ax.set_title(class_names[labels[i]], fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from model import get_model

model     = get_model(num_classes=len(class_names)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────
train_losses, val_losses   = [], []
train_accs,   val_accs     = [], []
best_val_acc               = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        t_loss    += loss.item() * imgs.size(0)
        t_correct += out.argmax(1).eq(lbls).sum().item()
        t_total   += imgs.size(0)

    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, lbls)
            v_loss    += loss.item() * imgs.size(0)
            v_correct += out.argmax(1).eq(lbls).sum().item()
            v_total   += imgs.size(0)

    t_acc = t_correct / t_total
    v_acc = v_correct / v_total
    train_losses.append(t_loss / t_total)
    val_losses.append(v_loss / v_total)
    train_accs.append(t_acc)
    val_accs.append(v_acc)
    scheduler.step()

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), MODEL_OUT)
        print(f'Epoch {epoch:3d} | Train {t_acc:.2%} | Val {v_acc:.2%}  ← best saved')
    else:
        print(f'Epoch {epoch:3d} | Train {t_acc:.2%} | Val {v_acc:.2%}')

print(f'\nBest Val Acc: {best_val_acc:.2%}  |  Model saved → {MODEL_OUT}')

In [ ]:
# ── Plot training curves ──────────────────────────────────────────
epochs_range = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(epochs_range, train_losses, label='Train Loss', color='#7c5cbf')
ax1.plot(epochs_range, val_losses,   label='Val Loss',   color='#ec4899')
ax1.set_title('Loss'); ax1.legend(); ax1.set_xlabel('Epoch')

ax2.plot(epochs_range, train_accs, label='Train Acc', color='#22c55e')
ax2.plot(epochs_range, val_accs,   label='Val Acc',   color='#f59e0b')
ax2.set_title('Accuracy'); ax2.legend(); ax2.set_xlabel('Epoch')

plt.suptitle('Training Curves — SkinCare CNN', fontsize=13)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Curves saved → training_curves.png')

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in val_loader:
        imgs = imgs.to(DEVICE)
        out  = model(imgs)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(lbls.numpy())

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=class_names))